# 04 拉格朗日松弛 —— 对偶函数与次梯度的完整推导

## 松弛覆盖约束

原问题 $\min\{\sum_p c_px_p:\ \sum_p a_{ip}x_p\ge 1,\ \sum_p x_p\le K,\ x_p\in\{0,1\}\}$。
给覆盖约束配乘子 $\lambda_i\ge 0$（惩罚"未覆盖"），得对偶函数：

$$L(\lambda)=\min_{\Sigma x\le K,\ x\in\{0,1\}}\Big[\sum_p c_px_p+\sum_i\lambda_i\Big(1-\sum_p a_{ip}x_p\Big)\Big]
=\sum_i\lambda_i+\min_{\Sigma x\le K}\sum_p\Big(\underbrace{c_p-\sum_i a_{ip}\lambda_i}_{rc_p}\Big)x_p$$

**子问题（平凡，无需求解器）**：按 $rc_p$ 排序取最负的至多 $K$ 个列（$rc\ge 0$ 不取）。

**对偶界**：任意 $\lambda\ge 0$ 都有 $L(\lambda)\le$ 原最优值（松弛约束只会降低 min 值）；
$\max_{\lambda\ge 0}L(\lambda)$ 是对偶问题（次梯度法求解）。

## 次梯度是怎么来的

**命题**：$g_i=1-\sum_p a_{ip}x_p(\lambda)$（$x(\lambda)$ 为 $L(\lambda)$ 的子问题解）是 $L$ 在 $\lambda$ 处的**次梯度**。

**证明**（对任意 $\lambda'\ge 0$，由子问题最优性 $L(\lambda)$ 取到 $x(\lambda)$，而 $L(\lambda')$ 是其 min）：

$$L(\lambda')=\min_x\Big[\sum_p c_px_p+\sum_i\lambda'_i(1-\sum_p a_{ip}x_p)\Big]\le\sum_p c_px_p(\lambda)+\sum_i\lambda'_i(1-\sum_p a_{ip}x_p(\lambda))$$

$$=\Big[\sum_p c_px_p(\lambda)+\sum_i\lambda_i(1-\sum_p a_{ip}x_p(\lambda))\Big]+\sum_i(\lambda'_i-\lambda_i)(1-\sum_p a_{ip}x_p(\lambda))
=L(\lambda)+\sum_i g_i(\lambda'_i-\lambda_i)$$

即 $L(\lambda')\ge L(\lambda)+\langle g,\lambda'-\lambda\rangle$，$g$ 是凹函数 $L$ 的次梯度（超平面支撑）。
**注意 $g_i=1-$覆盖次数**（布尔标志不是合法次梯度——02/04 家族实测的坑）。

## 为什么对偶 = LP 松弛值（integrality property）

子问题 $\min\{\sum rc_px_p:\Sigma x\le K,\ x_p\in\{0,1\}\}$ 的 LP 松弛（$x_p\in[0,1]$）最优值相同
（都等于取最负的至多 $K$ 个 rc 之和）。子问题具有"整点性质"⇒ 拉格朗日对偶
$\max L(\lambda)$ = 原问题的 LP 松弛值 = **191.813620**（02 已证），对偶间隙为零。


## 次梯度算法与实现要点

1. 迭代：$\lambda_i\leftarrow\max(0,\ \lambda_i+\alpha g_i)$，$\alpha=\rho\,(UB-L(\lambda))/\|g\|^2$；
   $\rho=2.0$ 起、30 轮无下界改进减半；**步长截断 100、λ 截断 1000**（$\|g\|\to 0$ 时步长发散）。
2. 修复：贪心给未覆盖客户补列（≤K 列）得上界；最后候选池 CP-SAT MIP 修复。
3. 停机：600 轮 / ρ<1e-5 / 墙钟 110s。确定性。
4. 与 02 的联系：取 $\lambda=\pi^*$（LP 最优对偶）时 $rc_p\ge 0$ 对全部列成立 ⇒ 子问题不选任何列 ⇒
   $L(\pi^*)=\sum_i\pi_i^*$ = LP 对偶目标 = LP 值 —— 强对偶点。


In [1]:
# 环境与演示数据（28 列小池 = 25 条单客户路径 + 3 条最优路线）
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="numpy")
import sys, platform, math, time
import ortools
sys.path.insert(0, "/mnt/d/exactTest/column-generation-solvers/vrptw_solomon25/scripts")
from cg_cpsat import build_data
from ortools.math_opt.python import mathopt
print("python", platform.python_version(), "| ortools", ortools.__version__)

n, xc, yc, dem, ready, due, svc, cap, depot_due, dist, d_scaled = build_data()
K = 25
ROUTES = [(13,17,18,19,15,16,14,12), (20,24,25,23,22,21), (5,3,7,8,10,11,9,6,4,2,1)]
pool = [(i,) for i in range(1, n+1)] + ROUTES

def col_cost(p):
    c = 0.0
    prev = 0
    for j in p:
        c += dist(prev, j)
        prev = j
    return c + dist(prev, 0)

def col_mask(p):
    m = 0
    for j in p:
        m |= (1 << j)
    return m

costs = [col_cost(p) for p in pool]
masks = [col_mask(p) for p in pool]
P = len(pool)
print(f"演示池: {P} 列（25 单客户 + 3 最优路线）；最优值 191.813620（=191.81）")


# 先解演示池 RMP 取对偶 π（作为 λ）
m = mathopt.Model()
xv = [m.add_variable(lb=0.0, ub=float("inf"), is_integer=False, name=f"x{p}") for p in range(P)]
covers = []
for i in range(1, n+1):
    covers.append(m.add_linear_constraint(
        mathopt.fast_sum([xv[p] for p in range(P) if (masks[p] >> i) & 1]) >= 1.0, name=f"c{i}"))
m.add_linear_constraint(mathopt.fast_sum(xv) <= K, name="veh")
m.minimize(mathopt.fast_sum([costs[p]*xv[p] for p in range(P)]))
res = mathopt.solve(m, mathopt.SolverType.GLOP)
dv = res.dual_values()
lam = [0.0]*(n+1)
for i in range(1, n+1):
    lam[i] = max(0.0, dv[covers[i-1]])
lp_val = res.objective_value()
print("LP 值 =", round(lp_val, 6), "| λ = LP 对偶 π")

# L(λ)：排序子问题
rcs = []
for p in range(P):
    s = 0.0
    mm = masks[p]
    while mm:
        lb = mm & -mm
        s += lam[lb.bit_length()-1]
        mm -= lb
    rcs.append((costs[p]-s, p))
rcs.sort()
S = [p for rc, p in rcs[:K] if rc < -1e-7]
L = sum(lam[1:]) + sum(costs[p]-sum(lam[i] for i in pool[p]) for p in S)
print(f"子问题选中 {len(S)} 列 | L(λ) = {round(L, 6)} | 与 LP 值相等: {abs(L-lp_val) < 1e-6}")
print("（λ=最优对偶时 rc_p≥0 对全部列成立，子问题不选列，L=Σλ=LP 对偶目标——强对偶点）")

# 次梯度（覆盖次数）与支撑性质验证（λ'=0）
cnt = [0]*(n+1)
for p in S:
    mm = masks[p]
    while mm:
        lb = mm & -mm
        cnt[lb.bit_length()-1] += 1
        mm -= lb
g = [1.0 - cnt[i] for i in range(1, n+1)]
print("次梯度 g = 1 - 覆盖次数 =", g[:6], "...")
L0 = 0.0   # λ'=0：rc=c>0，子问题不选列 -> L(0)=0
rhs = L + sum(g[i-1]*(0.0 - lam[i]) for i in range(1, n+1))
print(f"支撑性质: L(0)={L0} >= L(λ)+Σg·(0-λ) = {round(rhs,6)} -> {L0 >= rhs - 1e-9}")


python 3.10.20 | ortools 9.15.6755
python 3.10.20 | ortools 9.15.6755
演示池: 28 列（25 单客户 + 3 最优路线）；最优值 191.813620（=191.81）
LP 值 = 191.81362 | λ = LP 对偶 π
子问题选中 0 列 | L(λ) = 191.81362 | 与 LP 值相等: True
（λ=最优对偶时 rc_p≥0 对全部列成立，子问题不选列，L=Σλ=LP 对偶目标——强对偶点）
次梯度 g = 1 - 覆盖次数 = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0] ...
支撑性质: L(0)=0.0 >= L(λ)+Σg·(0-λ) = 0.0 -> True
